# SQL 기초 — 관계형 데이터베이스와 기본 조회

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# SQL 기초 — 관계형 데이터베이스와 기본 조회

## 테이블·스키마 · SELECT · FROM · WHERE · 연산자 · NULL

---

지금까지 우리는 pandas로 데이터를 불러와 분석했습니다. 그런데 현업에서 데이터는 대부분 **데이터베이스(Database)** 안에 살고 있습니다. 수백만 건의 주문, 수십만 명의 고객 정보가 거대한 표 더미로 저장되어 있습니다.

그 데이터에게 _"9월에 결제된 주문만 보여줘"_, _"미국 고객은 누구야?"_ 라고 **직접 묻는 언어**가 바로 **SQL(Structured Query Language)** 입니다.

오늘은 ① 데이터가 **어디에 어떻게** 저장되는지(데이터베이스)부터 시작해, ② SQL로 원하는 **열**을 고르고(`SELECT`), ③ 원하는 **행**을 거르고(`WHERE`), ④ 조건을 정교하게 표현하는 **연산자**까지 배웁니다.

> 💬 **오늘의 한 문장**  
> "SQL은 데이터에게 질문하는 언어다. 좋은 분석은 좋은 질문에서 시작한다."

## 📋 학습 목표

이 학습 노트북을 마치면 여러분은 다음을 할 수 있습니다.

1. 데이터베이스·관계형 데이터베이스·테이블·행·열·스키마·데이터 타입의 기본 개념을 설명할 수 있습니다.
2. `SELECT`·`FROM`으로 원하는 **열**을, `WHERE`로 원하는 **행**을 조회할 수 있습니다.
3. 비교·논리(`AND`·`OR`·`NOT`·`IN`·`BETWEEN`)·산술 연산자로 조건을 정교하게 표현할 수 있습니다.
4. 빈 값(`NULL`)을 `IS NULL`로 올바르게 다룰 수 있습니다.

_💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해 봅시다. 노트북을 마친 뒤 다시 돌아와 비교하면 성장이 눈에 보일 것입니다._

## 📚 학습 목차

| Part          | 주제                                           | 이 파트에서 답할 질문                    | 예상 시간            |
| ------------- | ---------------------------------------------- | ---------------------------------------- | -------------------- |
| **0**         | 실습 환경과 데이터 준비                        | 두 환경에서는 어떻게 실행합니까?         | 20분                 |
| **1**         | 관계형 데이터베이스와 테이블 구조              | 데이터베이스·테이블·스키마란 무엇입니까? | 25분                 |
| **2**         | 열 선택 — `SELECT` · `FROM`                    | 필요한 열만 어떻게 조회합니까?           | 40분                 |
| **3**         | 행 필터링 — `WHERE`와 비교 연산자              | 조건에 맞는 행만 어떻게 추립니까?        | 45분                 |
| **4**         | 논리 연산자 — `AND`·`OR`·`NOT`·`IN`·`BETWEEN`  | 여러 조건을 어떻게 결합합니까?           | 55분                 |
| **5**         | `NULL` 처리와 산술 연산자                      | `NULL`은 어떻게 다룹니까?                | 50분                 |
| **연습 문제** | 조회와 필터링 10문항                           | 배운 문법을 반복해서 손에 익힙니다       | 35분                 |
| **종합 실습** | 조건별 주문 조회 쿼리 작성                     | 조건을 조합해 직접 조회합니다            | 30분                 |
| **정리**      | 오늘의 퀴즈 · 개념 연결 도식 · 치트시트 · 과제 | 오늘 배운 것을 확인하고 정리합니다       | 20분                 |
|               | _합계_                                         |                                          | _320분 (5시간 20분)_ |

_⏱️ 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다._

# Part 0. 오늘의 여정

우리는 가상의 온라인 쇼핑몰 **"모두몰(MooduMall)"** 의 데이터 분석가입니다. 모두몰에는 신발·잡화를 사는 고객들이 있고, 매일 주문이 쌓입니다. 이 데이터는 **데이터베이스**에 저장되어 있습니다.

지금까지처럼 `pandas`로 CSV를 통째로 불러올 수도 있지만, 데이터가 수백만 건이라면 그 방법은 느리고 비효율적입니다. 그래서 분석가는 데이터베이스에 **SQL로 "필요한 것만" 질문**해서 가져옵니다.

> ❓ **질문하기**  
> pandas를 사용할 수 있는데도 SQL이 필요한 이유는 무엇입니까?

**SQL이 필요한 이유**

- **데이터가 사는 곳이 DB입니다.** 회사의 주문·회원·로그는 거의 다 데이터베이스에 있습니다. 분석의 출발점이 SQL인 경우가 많습니다.
- **필요한 범위만 요청합니다.** 데이터베이스에서 조건에 맞는 행과 열을 먼저 추리면 분석 환경으로 전송하고 처리할 데이터의 양을 줄일 수 있습니다.
- **핵심 문법을 폭넓게 활용할 수 있습니다.** `SELECT`·`FROM`·`WHERE` 같은 기본 문법은 여러 데이터베이스에서 거의 같은 형태로 사용합니다. 다만 데이터 타입과 일부 함수·관리 명령은 제품마다 다를 수 있습니다.

> 💡 **핵심짚기 — 공통 문법과 DBMS별 방언을 구분합니다**  
> 이 교안은 LMS(DuckDB)와 BigQuery에서 공통으로 사용할 수 있는 조회 문법을 중심으로 작성했습니다. 환경별 차이가 있는 문법은 따로 표시합니다.

## ⚙️ 환경 설정과 실습 데이터

> ⚠️ **주의하기 — 셀은 위에서부터 순서대로 실행**  
> 코드 셀은 반드시 위에서부터 순서대로 실행합니다. 중간부터 실행하면 앞에서 만들어야 할 스키마나 테이블이 없어 오류가 발생할 수 있습니다.

아래 준비 셀을 실행하면 SQL 실습에 필요한 이름 공간이 생성됩니다. 이 단계에서는 구문의 세부 내용보다 앞에서부터 순서대로 실행해야 한다는 점을 확인합니다.

## 실습 환경 — LMS 또는 BigQuery

이 노트북의 SQL은 **LMS 실습 환경(DuckDB)** 또는 **BigQuery** 어느 쪽에서든 실행되도록 작성했습니다. 둘 중 **편한 쪽 하나만** 고르면 됩니다.

- **LMS에서 학습한다면** — 바로 아래 `[C1]` 준비 셀을 먼저 실행한 뒤, 이어지는 셀을 위에서부터 순서대로 실행합니다.
- **BigQuery에서 학습한다면** — `[C1]`은 건너뛰고 각 셀의 SQL을 BigQuery 콘솔이나 샌드박스에서 실행합니다. 테이블 이름의 `project_name.dataset_name`은 자신의 프로젝트·데이터셋 이름으로 변경합니다. 실제 프로젝트 ID에 하이픈(`-`)이 있으면 전체 경로를 `` `my-project.dataset_name.table_name` ``처럼 백틱으로 감쌉니다.

> ⚠️ **주의하기 — 환경에 따라 일부 문법이 다릅니다**  
> 기본 조회 문법은 대부분 같지만 데이터 타입, 날짜 함수, 메타데이터 확인 명령 등은 다를 수 있습니다. 차이가 있는 셀의 안내를 확인하고 자신의 환경에 맞는 문법을 사용합니다.

## 두 환경의 문법 차이 — 한눈에 보기

| 항목              | LMS (DuckDB)                   | BigQuery                           | 교안 적용 방식                           |
| ----------------- | ------------------------------ | ---------------------------------- | ---------------------------------------- |
| 테이블 경로       | `project_name.dataset_name.표` | `project_name.dataset_name.표`     | `[C1]`이 LMS에 같은 이름 공간을 만듭니다 |
| 문자열 타입       | `VARCHAR` (`STRING`도 허용)    | `STRING`                           | 생성 셀에서는 `STRING`을 사용합니다      |
| 정수 타입         | `INTEGER`·`BIGINT`             | `INT64`                            | 환경에 맞는 타입 이름을 사용합니다       |
| 고정 소수점 타입  | `DECIMAL(12,2)`                | `NUMERIC(12,2)` (`DECIMAL`도 허용) | 금액에는 고정 소수점 타입을 사용합니다   |
| 월 단위 날짜 절단 | `date_trunc('month', d)`       | `DATE_TRUNC(d, MONTH)`             | 차이가 있는 셀에 두 문법을 안내합니다    |
| 날짜 차이         | `date_diff('day', 시작, 끝)`   | `DATE_DIFF(끝, 시작, DAY)`         | 인자 순서까지 확인합니다                 |
| 테이블 구조 확인  | `DESCRIBE 테이블`              | `INFORMATION_SCHEMA.COLUMNS` 조회  | `[C5]`에서 환경별 방법을 안내합니다      |

> 💡 **핵심짚기 — 공통 문법과 방언을 구분합니다**  
> `SELECT`·`FROM`·`WHERE` 같은 핵심 조회 문법은 두 환경에서 동일하게 사용할 수 있습니다. 반면 함수·데이터 타입·관리 명령에는 데이터베이스별 방언이 있으므로, 실무에서는 사용하는 DBMS의 문서를 함께 확인합니다.

> 📌 **실무 연결하기 — DBMS가 바뀌면 방언을 확인합니다**  
> 같은 분석 로직이라도 날짜 함수나 데이터 타입 표기는 달라질 수 있습니다. 쿼리를 다른 DBMS로 옮길 때는 함수의 이름·인자 순서·반환 타입을 먼저 점검하는 습관이 필요합니다.

In [ ]:
-- [C1] LMS(DuckDB) 준비 — BigQuery로 학습한다면 이 셀은 건너뜁니다.
-- BigQuery와 똑같은 `project_name.dataset_name.표` 이름을 쓸 수 있도록 이름 공간을 만듭니다.
ATTACH IF NOT EXISTS ':memory:' AS project_name;
CREATE SCHEMA IF NOT EXISTS project_name.dataset_name;

## 모두몰 데이터 만들기

분석할 데이터를 데이터베이스에 만들어 둡니다. 두 개의 **테이블(표)** 을 만들 겁니다.

- **`customers`** — 고객 정보 (고객번호·이름·국가·가입일·등급)
- **`orders`** — 주문 정보 (주문번호·고객번호·주문일·상태·금액)

아래 셀의 `CREATE TABLE`(표 만들기)과 `INSERT`(값 넣기)는 **데이터를 준비하는 코드**입니다. 오늘 배울 핵심은 이 다음부터입니다.

In [ ]:
-- [C2]
CREATE OR REPLACE TABLE project_name.dataset_name.customers (
  customer_id STRING, /* 고객번호 (예: C001) */
  name STRING, /* 이름 */
  country STRING, /* 국가 */
  signup_date DATE, /* 가입일 */
  grade STRING /* 등급 (Gold/Silver/Bronze) */
);

INSERT INTO project_name.dataset_name.customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')

In [ ]:
-- [C3]
CREATE OR REPLACE TABLE project_name.dataset_name.orders (
  order_id STRING, /* 주문번호 */
  customer_id STRING, /* 고객번호 (customers와 연결) */
  order_date DATE, /* 주문일 */
  status STRING, /* 상태 (Placed/Paid/Shipped/Cancelled/Returned) */
  amount DECIMAL(12,2) /* 결제 금액 (취소 시 비어 있을 수 있음) */
);

INSERT INTO project_name.dataset_name.orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000)

In [ ]:
-- [C4] customers 테이블에 데이터가 들어갔는지 결과 중 5행을 확인합니다.
SELECT
  *
FROM project_name.dataset_name.customers
LIMIT 5

> 💡 **핵심짚기 — `LIMIT`은 행의 순서를 정하지 않습니다**  
> 위의 `LIMIT 5`는 결과에서 5행만 보여줍니다. 정렬 기준을 지정하지 않았으므로 특정한 '첫 5행'을 보장하지는 않습니다. 지금은 데이터가 들어갔는지만 확인하고, 원하는 순서를 정하는 `ORDER BY`는 다음 시간에 배웁니다.

자, 이제 **이 데이터에게 질문을 던질** 준비가 됐습니다. 그 전에, 이 데이터가 *어떻게 생긴 그릇*에 담겨 있는지부터 알아보겠습니다.

# Part 1. 관계형 데이터베이스와 테이블 구조

SQL로 질문하기 전에, 데이터가 _어떤 모양으로_ 저장되어 있는지부터 알아야 합니다. 방금 만든 `customers`·`orders`가 바로 그 모양 — **테이블** 입니다. 이 Part에서는 데이터베이스의 가장 기본이 되는 용어들을 차근차근 정리합니다.

> ❓ **질문하기**  
> 데이터베이스·관계형 데이터베이스·테이블·행·열·스키마·데이터 타입은 각각 무엇입니까?

## 🧩 개념 잡기

처음에는 데이터베이스를 **여러 표를 체계적으로 관리하는 공간**이라고 이해하면 좋습니다. 엑셀 파일과 비슷해 보이지만, 데이터베이스는 여러 사용자의 동시 작업, 대용량 조회, 권한 관리, 무결성 규칙 등을 DBMS가 관리한다는 점에서 차이가 있습니다.

```
데이터베이스 (MooduMall)
 ├─ customers  테이블            ← 고객 명단
 └─ orders     테이블            ← 주문 장부
```

각 **테이블**은 엑셀 시트처럼 **행(Row)** 과 **열(Column)** 로 이루어져 있습니다.

```
        ┌── 열(column): 같은 종류의 값 (예: 모든 고객의 '국가')
        ▼
   customer_id │ name   │ country
   ────────────┼────────┼─────────
   C001        │ 김민준 │ Korea     ◄── 행(row): 고객 한 명의 모든 정보
   C002        │ 이서연 │ Korea
```

## 데이터와 데이터베이스 — 왜 따로 모아둘까

**데이터(Data)** 는 "기록된 사실"입니다. 누가·언제·무엇을 샀는지 같은 것들입니다. 이런 데이터가 쌓이면 어딘가에 **안전하고 찾기 쉽게** 보관해야 합니다.

처음엔 엑셀 파일 하나로 충분할 수 있습니다. 하지만 데이터가 커지면 엑셀로는 한계가 옵니다.

| 엑셀 파일로 관리하면                              | 데이터베이스로 관리하면                                |
| ------------------------------------------------- | ------------------------------------------------------ |
| 데이터가 커지면 파일을 열고 처리하기 부담스럽다   | 인덱스·분산 처리 등으로 대용량 조회를 효율화할 수 있다 |
| 여러 명이 동시에 수정할 때 충돌을 관리하기 어렵다 | 트랜잭션과 권한으로 동시 작업을 체계적으로 관리한다    |
| 조건에 맞는 데이터만 반복해서 찾기 번거롭다       | SQL로 필요한 범위와 열을 명확하게 요청한다             |
| 입력 형식과 값의 규칙을 일관되게 강제하기 어렵다  | 타입·제약 조건으로 데이터 품질을 관리할 수 있다        |

이렇게 데이터를 체계적으로 저장·관리하는 시스템을 **데이터베이스(Database)** 라 하고, 이를 운영하는 소프트웨어를 **DBMS(Database Management System)** 라고 합니다. 우리가 SQL로 말을 거는 상대가 바로 이 DBMS입니다.

> 🏷️ **용어 — SQL(Structured Query Language)**  
> 이 데이터베이스에게 _질문하고 다루는_ 표준 언어입니다. 오늘 배우는 건 그중에서도 데이터를 **조회**(질문)하는 부분입니다.

## 관계형 데이터베이스 — 표를 나눠 담는 이유

널리 사용되는 데이터베이스 유형 중 하나가 **관계형 데이터베이스(Relational Database, RDB)** 입니다. 관계형 모델에서는 데이터를 행과 열로 이루어진 **관계(relation)** 로 표현하며, 실무에서는 이를 테이블이라고 부릅니다. 관련 있는 테이블은 고객번호처럼 공통 의미를 가진 열로 연결해 조회할 수 있습니다.

테이블을 나누는 주된 이유 중 하나는 **같은 정보의 불필요한 중복을 줄이는 것**입니다.

```
[나쁜 예] 주문 표에 고객 정보까지 다 넣으면…
 주문번호 │ 고객이름 │ 고객국가 │ 상품
 O0001    │ 김민준   │ Korea    │ 운동화   ← 김민준이 10번 사면
 O0011    │ 김민준   │ Korea    │ 부츠      '김민준/Korea'가 10번 반복!

[좋은 예] 고객은 customers에 한 번만, 주문은 orders에 — 번호로 연결
 customers: C001 │ 김민준 │ Korea
 orders:    O0001 │ C001 │ 운동화      ← 'C001'만 적으면 됨
```

이렇게 나눠 담으면 중복이 줄고, 고객 이름이 바뀌어도 **한 군데만** 고치면 됩니다. 표를 잇는 구체적인 방법(`JOIN`)은 이후 배웁니다. 오늘은 _"데이터가 여러 표로 나뉘어 있다"_ 는 사실만 기억하면 됩니다.

> 📌 **실무 연결하기 — 산업마다 같은 테이블 분리 구조**  
> 여러 산업의 관계형 데이터베이스에서 비슷한 분리 구조를 볼 수 있습니다. 예를 들어 금융은 계좌·거래 내역, 병원은 환자·진료 기록, 학교는 학생·수강 신청 데이터를 별도 테이블로 관리하고 식별자로 연결할 수 있습니다.

## 테이블·행·열 — 표의 구성 요소

테이블을 이루는 세 가지를 정리하면 다음과 같습니다.

| 용어             | 의미                          | 모두몰 예시              |
| ---------------- | ----------------------------- | ------------------------ |
| 테이블(Table)    | 행과 열로 된 표 하나          | `customers`, `orders`    |
| 행(Row, 레코드)  | 가로 한 줄 = 대상 하나의 정보 | 고객 'C001 김민준' 한 명 |
| 열(Column, 필드) | 세로 한 줄 = 한 종류의 값     | `country`(국가) 열       |

> 🐼 **pandas와 비교**  
> 사실 여러분은 이미 이 구조를 알고 있습니다!
>
> - SQL의 **테이블** = pandas의 **`DataFrame`**
> - SQL의 **행** = `DataFrame`의 한 행(row)
> - SQL의 **열** = `DataFrame`의 **column**
>
> 이름만 다를 뿐 같은 개념입니다. SQL은 이 표에게 _문장으로_ 질문하는 방법입니다.

## 스키마 — 테이블의 설계도

이 교안에서는 **테이블 스키마(table schema)** 를 테이블의 구조 설계도라는 뜻으로 사용합니다. 어떤 열이 있고 각 열의 데이터 타입과 제약 조건이 무엇인지 정의한 정보입니다. 앞의 `CREATE TABLE` 문에 적은 열 이름과 타입이 이에 해당합니다.

구조를 확인하는 방법은 DBMS마다 다릅니다. LMS(DuckDB)에서는 `DESCRIBE`를 사용하고, BigQuery에서는 `INFORMATION_SCHEMA.COLUMNS`를 조회합니다.

In [ ]:
-- [C5] LMS(DuckDB): customers 테이블의 구조를 확인합니다.
DESCRIBE project_name.dataset_name.customers;

-- BigQuery에서는 위 문장 대신 아래 주석의 SELECT 문을 사용합니다.
-- SELECT column_name, data_type
-- FROM `project_name`.dataset_name.INFORMATION_SCHEMA.COLUMNS
-- WHERE table_name = 'customers'
-- ORDER BY ordinal_position;

> 👀 **결과읽기 — [C5]**  
> 결과의 열 이름 항목에는 `customer_id`·`name`·`country`·`signup_date`·`grade`가 나타납니다. 타입 항목에서는 문자열 열과 `DATE` 열을 구분할 수 있습니다. 타입 이름과 결과 열 이름은 DBMS에 따라 `STRING`/`VARCHAR`, `data_type`/`column_type`처럼 다르게 표시될 수 있습니다.

## 데이터 타입 — 값의 종류를 정해두는 이유

각 열에는 **어떤 종류의 값이 들어갈지(데이터 타입)** 가 정해져 있습니다. 자주 쓰는 타입은 다음과 같습니다.

| 값의 종류      | LMS(DuckDB) 예     | BigQuery 예 | 값 예시              |
| -------------- | ------------------ | ----------- | -------------------- |
| 문자열         | `VARCHAR`          | `STRING`    | `'김민준'`, `'Paid'` |
| 정수           | `INTEGER`·`BIGINT` | `INT64`     | `42`, `1000`         |
| 고정 소수점 수 | `DECIMAL`          | `NUMERIC`   | `125000.50`          |
| 날짜           | `DATE`             | `DATE`      | `DATE '2023-09-02'`  |
| 참·거짓        | `BOOLEAN`          | `BOOL`      | `TRUE`, `FALSE`      |

**데이터 타입이 중요한 이유**는 데이터베이스가 값의 의미에 맞게 계산·정렬·비교해야 하기 때문입니다. 예를 들어 `DATE` 타입은 날짜 차이 계산을 지원하지만, 같은 모양의 문자열에는 날짜 연산을 바로 적용할 수 없습니다. 또한 DBMS는 타입과 제약 조건에 맞지 않는 값의 저장을 거부하거나 변환합니다. 구체적인 변환 규칙은 DBMS마다 다릅니다.

> ⚠️ **주의하기 — 문자열은 작은따옴표로 감싼다**  
> 문자열 값은 SQL에서 **작은따옴표**로 감쌉니다 — `'Paid'`. 숫자는 따옴표 없이 `125000`. 이 구분은 앞으로 계속 나옵니다.

## ⌨️ 백문이 불여일타 (1)

```
[문제] `orders` 테이블의 스키마를 확인하고 각 열의 데이터 타입을 살펴봅니다.
```

In [ ]:
-- [C6] ⌨️ 백문이 불여일타 (1) — orders 테이블 스키마 확인

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- LMS에서는 Part 1의 `[C5]`와 같은 `DESCRIBE` 문에서 테이블 이름만 `orders`로 바꿉니다.
- BigQuery에서는 `[C5]`의 `INFORMATION_SCHEMA.COLUMNS` 조회에서 `table_name` 조건을 `'orders'`로 바꿉니다.
- 테이블 이름은 `project_name.dataset_name.orders`처럼 전체 경로로 적습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

**LMS(DuckDB)**

```sql
DESCRIBE project_name.dataset_name.orders;
```

**BigQuery**

```sql
SELECT column_name, data_type
FROM `project_name`.dataset_name.INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'orders'
ORDER BY ordinal_position;
```

</details>

> ❓ **질문하기**  
> `amount` 열을 고정 소수점 타입(`DECIMAL`/`NUMERIC`)으로 저장하면 어떤 장점이 있습니까? `order_date`를 `DATE` 타입으로 저장하면 어떤 날짜 연산이 쉬워집니까?

## 🚦 짚고 넘어가기

다음 질문으로 핵심 개념을 확인한 뒤 다음 Part로 넘어갑니다.

1. 데이터베이스가 스프레드시트 파일보다 적합한 상황을 두 가지 설명합니다.
2. 관계형 데이터베이스가 데이터를 _여러 표로 나눠_ 담는 이유는?
3. 테이블에서 '행'과 '열'은 각각 무엇을 담습니까?
4. 데이터 타입을 정의하면 어떤 장점이 있습니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 여러 사람이 동시에 안전하게 사용할 수 있고, 아주 큰 데이터도 문제없이 다룰 수 있습니다. (규칙이 강제되어 데이터가 잘 깨지지 않는 것도 장점입니다.)
2. 같은 정보가 여러 곳에 중복 저장되는 것을 막고, 수정할 때 한 곳만 고치면 되게 하기 위해서입니다.
3. 행은 데이터 한 건(예: 고객 한 명)을, 열은 그 데이터의 속성(예: 이름·국가)을 담습니다.
4. 잘못된 값이 들어오는 것을 막아주고, 타입에 맞는 연산(날짜 비교·숫자 계산)을 안심하고 할 수 있습니다.

</details>

> ⏭️ **다음 학습 예고**  
> 데이터의 구조를 확인했으므로 이제 `SELECT`로 **필요한 열만 조회하는 방법**을 알아봅니다.

# Part 2. 열 선택 — `SELECT` · `FROM`

데이터를 조회하는 쿼리는 `SELECT`로 시작합니다. `SELECT`는 결과에 어떤 값을 표시할지 정하는 절입니다.

> ❓ **질문하기**  
> 테이블에서 원하는 열만 어떻게 조회합니까?

## 🧩 개념 잡기

`SELECT`는 식당에서 **주문하는 것** 과 같습니다.

```
SELECT  name, grade     ← 무엇을 줄까? (가져올 '열' 고르기)
FROM    customers       ← 어디에서?   (대상 '테이블')
```

즉, "`customers` 테이블에서 `name`과 `grade` 열을 조회합니다"라는 의미입니다.

## `SELECT` ... `FROM` ... 기본 문법

```sql
SELECT 열1, 열2, ...      -- 보고 싶은 열 (전체는 *)
FROM   테이블이름          -- 어느 테이블에서
```

- `SELECT *` : 모든 열을 가져옵니다 (`*`는 '전부'라는 뜻).
- `SELECT name, country` : 지정한 열만 가져옵니다.

> ⚠️ **주의하기 — 결과 행의 순서는 보장되지 않습니다**  
> `SELECT`에 열을 적는 순서는 결과의 **열 순서**를 정하지만, 결과의 **행 순서**는 정하지 않습니다. 행을 일정한 순서로 보려면 `ORDER BY`가 필요합니다. 따라서 `ORDER BY` 없는 `LIMIT 5`는 특정한 첫 5행이 아니라 결과 중 5행을 확인하는 용도입니다.

> 🐼 **pandas와 비교**
>
> - `SELECT * FROM customers` ↔ `df`
> - `SELECT name, country FROM customers` ↔ `df[['name', 'country']]`

▶️ **코드 실행하기 · 코드 셀 7 [C7]**  
— `customers`에서 **이름·국가·등급 세 열만** 골라 조회합니다.

In [ ]:
-- [C7]
SELECT
  name,
  country,
  grade
FROM project_name.dataset_name.customers

> 👀 **결과읽기 — [C7]**  
> `[C4]`의 `SELECT *`는 모든 열을 조회했지만, `[C7]`은 `name`·`country`·`grade` 세 열만 조회합니다. 필요한 열만 명시하면 결과의 의미가 분명해지고 불필요한 데이터 전송도 줄일 수 있습니다.

## ⌨️ 백문이 불여일타 (2)

```
[문제] `orders` 테이블에서 주문번호(`order_id`), 주문일(`order_date`), 금액(`amount`) 세 열만 조회합니다.
```

In [ ]:
-- [C8] ⌨️ 백문이 불여일타 (2) — orders에서 세 열만 골라 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 열을 고르는 절은 `SELECT`, 테이블을 지정하는 절은 `FROM`입니다.
- `SELECT` 뒤에 원하는 열 이름을 쉼표로 구분해 나열합니다.
- 테이블은 `project_name.dataset_name.orders`입니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  order_date,
  amount
FROM project_name.dataset_name.orders
```

</details>

## ⌨️ 백문이 불여일타 (3)

```
[문제] `customers`에서 가입일(`signup_date`)과 이름(`name`)을 가입일 → 이름 순서로 조회합니다.
```

In [ ]:
-- [C9] ⌨️ 백문이 불여일타 (3) — customers 열 순서 바꿔 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- `SELECT`에 적는 순서가 곧 결과 열 순서입니다.
- 가입일을 먼저 보고 싶으면 `signup_date`를 `name`보다 앞에 적으면 됩니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  signup_date,
  name
FROM project_name.dataset_name.customers
```

</details>

## 🚦 짚고 넘어가기

1. `SELECT *` 와 `SELECT name` 의 차이는 무엇입니까?
2. `FROM` 뒤에는 무엇이 옵니까?
3. `SELECT`에 적는 열 순서가 결과에 영향을 줍니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `SELECT *`는 모든 열을, `SELECT name`은 `name` 열 하나만 가져옵니다.
2. 조회할 테이블 이름(경로)이 옵니다.
3. 네 — `SELECT`에 적은 순서 그대로 결과 열이 나옵니다.

</details>

> ⏭️ **다음 학습 예고**  
> 지금은 모든 행이 조회됩니다. 이어서 `WHERE`로 **조건에 맞는 행만 필터링하는 방법**을 알아봅니다.

# Part 3. 행 필터링 — `WHERE`와 비교 연산자

`SELECT`가 *열*을 고른다면, `WHERE`는 *행*을 거릅니다. "조건에 맞는 행만 보여줘"라는 뜻입니다. SQL 분석의 진짜 시작입니다.

> ❓ **질문하기**  
> 수많은 행 중에서 조건에 맞는 행만 어떻게 추립니까?

## 🧩 개념 잡기

`WHERE`는 **체에 거르는 것** 과 같습니다. 전체 주문을 체에 부으면, 조건(`status = 'Paid'`)에 맞는 행만 아래로 통과합니다.

```
전체 주문 (30건)
   │   WHERE status = 'Paid'   ← 조건이라는 '체'
   ▼
결제된 주문만 (통과한 행)
```

## `WHERE` + 비교 연산자

`WHERE` 뒤에는 참·거짓을 판단할 **조건식**을 작성합니다. 조건식에는 다음과 같은 비교 연산자를 사용할 수 있습니다.

| 연산자         | 의미               | 예시                    |
| -------------- | ------------------ | ----------------------- |
| `=`            | 같다               | `status = 'Paid'`       |
| `!=` 또는 `<>` | 같지 않다          | `status != 'Cancelled'` |
| `>` , `>=`     | 크다 / 크거나 같다 | `amount >= 100000`      |
| `<` , `<=`     | 작다 / 작거나 같다 | `amount < 50000`        |

> ⚠️ **주의하기 — 문자열은 작은따옴표, 숫자는 그대로**  
> 문자 값은 작은따옴표로 감쌉니다 — `'Paid'` (O), `Paid` (X). 숫자는 따옴표 없이 — `100000`.

> ⚠️ **주의하기 — 비교 연산자는 `NULL`을 포함하지 않습니다**  
> `status <> 'Cancelled'` 같은 비교 결과는 값이 `NULL`이면 참이 아니라 `UNKNOWN`이 됩니다. 따라서 `WHERE` 결과에서 제외됩니다. `NULL` 포함 여부는 Part 5의 `IS NULL`·`IS NOT NULL`로 명시해야 합니다.

> 🐼 **pandas와 비교**  
> `SELECT * FROM orders WHERE amount >= 100000` ↔ `df[df['amount'] >= 100000]`

▶️ **코드 실행하기 · 코드 셀 10 [C10]**  
— 주문 중 **결제 완료(`Paid`) 상태인 것만** 골라 주문번호·상태·금액을 조회합니다.

In [ ]:
-- [C10]
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  status = 'Paid'

▶️ **코드 실행하기 · 코드 셀 11 [C11]**  
— 주문 중 **금액이 10만원 이상인 것만** 골라 조회합니다.

In [ ]:
-- [C11]
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount >= 100000

> 👀 **결과읽기 — [C11]**  
> `WHERE`를 붙이는 순간 30건 전체가 아니라 _조건에 맞는 행만_ 남습니다. 분석가는 이렇게 "보고 싶은 데이터"를 정확히 좁혀갑니다.

> 📌 **실무 연결하기 — 조건 필터링의 산업별 쓰임**  
> `WHERE` 한 줄이면 금융에서는 _이상거래(금액 > 1000만원)_, 제조에서는 _불량(status = 'defect')_, 헬스케어에서는 _고위험군(age >= 65)_ 만 즉시 추려낼 수 있습니다.

## ⌨️ 백문이 불여일타 (4)

```
[문제] `orders`에서 상태가 `Cancelled`가 아닌 주문의 주문번호·상태·금액을 조회합니다.
```

In [ ]:
-- [C12] ⌨️ 백문이 불여일타 (4) — 취소가 아닌 주문만 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- '같지 않다'는 `!=` 또는 `<>`로 씁니다.
- 행을 거르는 조건은 `WHERE` 절에 적습니다.
- 문자 값은 `'Cancelled'`처럼 따옴표로 감싸야 합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  status <> 'Cancelled'
```

</details>

## ⌨️ 백문이 불여일타 (5)

```
[문제] `orders`에서 금액이 5만원 미만인 주문의 주문번호·금액을 조회합니다.
```

In [ ]:
-- [C13] ⌨️ 백문이 불여일타 (5) — 5만원 미만 소액 주문 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- '미만'은 `<`, '이하'는 `<=`입니다.
- 숫자 조건에는 따옴표를 쓰지 않습니다 — `amount`와 50000을 그대로 비교하면 됩니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount < 50000
```

</details>

## 🚦 짚고 넘어가기

1. `WHERE`는 행을 거릅니까, 열을 거릅니까?
2. 문자 값과 숫자 값을 조건에 쓸 때 차이가 있습니까?
3. `amount > 100000` 과 `amount >= 100000` 의 차이는?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 행을 거릅니다. 열을 고르는 것은 `SELECT`입니다.
2. 네 — 문자 값은 `'Paid'`처럼 따옴표로 감싸고, 숫자 값은 그대로 씁니다.
3. `>`는 10만원을 포함하지 않고(초과), `>=`는 포함합니다(이상).

</details>

> ⏭️ **다음 학습 예고**  
> 조건이 하나면 `WHERE` 하나로 충분하지만, _"미국 고객이면서 Gold 등급"_ 처럼 **여러 조건을 묶고 싶을 때**는 논리 연산자가 필요합니다.

# Part 4. 논리 연산자 — `AND` · `OR` · `NOT` · `IN` · `BETWEEN`

실제 분석 질문에는 여러 조건이 함께 등장합니다. 예를 들어 "한국 고객 중에서 Gold 등급인 고객"은 국가와 등급 조건을 동시에 만족해야 합니다. 이때 `AND`·`OR`·`NOT`·`IN`·`BETWEEN`으로 조건을 결합합니다.

> ❓ **질문하기**  
> 여러 조건을 어떻게 하나의 조건식으로 결합합니까?

## 🧩 개념 잡기

- `AND` = **그리고** (두 조건 모두 만족) → 교집합
- `OR` = **또는** (둘 중 하나라도 만족) → 합집합
- `NOT` = **아니다** (조건을 뒤집기)

```
country='Korea' AND grade='Gold'   →  한국 사람 ∩ Gold 등급  (둘 다)
country='Korea' OR  country='USA'  →  한국 ∪ 미국           (둘 중 하나)
```

## `AND` / `OR` / `NOT`

| 연산자 | 의미           | 예시                                   |
| ------ | -------------- | -------------------------------------- |
| `AND`  | 모든 조건 만족 | `country = 'Korea' AND grade = 'Gold'` |
| `OR`   | 하나라도 만족  | `grade = 'Gold' OR grade = 'Silver'`   |
| `NOT`  | 조건 부정      | `NOT (country = 'USA')`                |

> 🐼 **pandas와 비교**  
> pandas에서는 `&`(AND), `|`(OR), `~`(NOT)를 사용합니다. SQL에서는 `AND`·`OR`·`NOT`이라는 키워드를 사용합니다.

> ⚠️ **주의하기 — `AND`가 `OR`보다 먼저 계산됩니다**  
> 두 연산자를 함께 사용할 때 의도가 분명하도록 괄호를 사용합니다. 예를 들어 `country = 'Korea' AND (grade = 'Gold' OR grade = 'Silver')`처럼 작성합니다.

▶️ **코드 실행하기 · 코드 셀 14 [C14]**  
— **한국 고객이면서 동시에 `Gold` 등급인** 고객만 조회합니다(`AND`).

In [ ]:
-- [C14]
SELECT
  name,
  country,
  grade
FROM project_name.dataset_name.customers
WHERE
  country = 'Korea' AND grade = 'Gold'

▶️ **코드 실행하기 · 코드 셀 15 [C15]**  
— **`Gold` 등급 또는 `Silver` 등급인** 고객을 조회합니다(`OR`).

In [ ]:
-- [C15]
SELECT
  name,
  country,
  grade
FROM project_name.dataset_name.customers
WHERE
  grade = 'Gold' OR grade = 'Silver'

## `IN` · `BETWEEN` — 더 간결하게

`OR`가 길어질 때, `BETWEEN`(범위), `IN`(목록)을 쓰면 훨씬 깔끔합니다.

| 연산자            | 의미                 | 길게 쓰면                   |
| ----------------- | -------------------- | --------------------------- |
| `BETWEEN a AND b` | a 이상 b 이하 (범위) | `x >= a AND x <= b`         |
| `IN (...)`        | 목록 중 하나         | `x = 'A' OR x = 'B' OR ...` |
| `NOT IN (...)`    | 목록에 없음          | —                           |

> 💡 `BETWEEN`은 양 끝값을 **포함**합니다.

▶️ **코드 실행하기 · 코드 셀 16 [C16]**  
— 주문 중 **금액이 5만원 이상 10만원 이하인** 것만 조회합니다 — `BETWEEN`은 양 끝값을 포함합니다.

In [ ]:
-- [C16]
SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount BETWEEN 50000 AND 100000

▶️ **코드 실행하기 · 코드 셀 17 [C17]**  
— 주문 중 **상태가 `Paid` 또는 `Shipped`인** 것만 조회합니다(`IN`).

In [ ]:
-- [C17]
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  status IN ('Paid', 'Shipped')

> 👀 **결과읽기 — [C17]**  
> `status IN ('Paid', 'Shipped')`는 `status = 'Paid' OR status = 'Shipped'`와 같은 조건을 더 간결하게 표현합니다. 값의 목록이 길수록 `IN`을 사용한 쿼리를 읽고 수정하기 쉽습니다.

> 📌 **실무 연결하기 — `IN`·`BETWEEN`의 산업별 쓰임**  
> 마케팅에서 _특정 캠페인 그룹(`campaign IN ('A','B','C')`)_, 제조에서 _허용 온도 범위(`temp BETWEEN 20 AND 25`)_, 금융에서 _관심 지점(`branch IN (...)`)_ 을 추릴 때 매일 쓰는 패턴입니다.

## ⌨️ 백문이 불여일타 (6)

```
[문제] `IN`을 사용해 `customers`에서 국가가 한국 또는 일본인 고객을 조회합니다.
```

In [ ]:
-- [C18] ⌨️ 백문이 불여일타 (6) — 한국·일본 고객 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 여러 값 중 하나에 해당하는지 볼 때는 `IN (값1, 값2)`가 간결합니다.
- 국가 이름은 문자이므로 `'Korea'`, `'Japan'`처럼 따옴표가 필요합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  name,
  country,
  grade
FROM project_name.dataset_name.customers
WHERE
  country IN ('Korea', 'Japan')
```

</details>

## ⌨️ 백문이 불여일타 (7)

```
[문제] `AND`와 `BETWEEN`을 사용해 `orders`에서 상태가 `Paid`이고 금액이 10만원 이상 30만원 이하인 주문을 조회합니다.
```

In [ ]:
-- [C19] ⌨️ 백문이 불여일타 (7) — Paid이면서 10만~30만원 주문 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 두 조건을 모두 만족해야 하므로 `AND`로 묶습니다.
- '10만원 이상 30만원 이하'는 `BETWEEN`으로 쓸 수 있습니다 — 양 끝 값을 포함합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  status = 'Paid' AND amount BETWEEN 100000 AND 300000
```

</details>

## 🚦 짚고 넘어가기

1. `AND`와 `OR`의 차이를 한 문장으로 설명합니다.
2. `amount BETWEEN 50000 AND 100000`은 5만원과 10만원을 포함합니까?
3. `status IN ('Paid', 'Shipped')`를 `OR` 조건으로 바꾸어 작성합니다.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `AND`는 두 조건을 모두 만족해야 하고, `OR`는 둘 중 하나만 만족해도 됩니다.
2. 네 — `BETWEEN`은 양 끝 값을 모두 포함합니다.
3. `status = 'Paid' OR status = 'Shipped'`이며, 현재 데이터에서는 같은 결과가 반환됩니다.

</details>

> ⏭️ **다음 학습 예고**  
> 취소된 주문처럼 값이 정해지지 않은 행에는 `NULL`이 저장될 수 있습니다. 이어서 `NULL`을 올바르게 판별하는 방법과 조회 과정에서 계산하는 방법을 알아봅니다.

# Part 5. `NULL` 처리와 산술 연산자

데이터에는 **비어 있는 값**이 흔합니다. 모두몰의 취소 주문은 결제가 안 됐으니 `amount`가 비어 있었습니다. SQL에서 이 빈 값은 `NULL`이라고 부르고, **특별하게** 다뤄야 합니다.

> ❓ **질문하기**  
> 값이 없거나 알려지지 않은 상태인 `NULL`은 어떻게 판별합니까?

## 🧩 개념 잡기

`NULL`은 **값이 없거나 알려지지 않은 상태**를 나타냅니다. 숫자 `0`이나 길이가 0인 문자열(`''`)도 하나의 값이므로 `NULL`과 다릅니다.

SQL의 조건식은 `TRUE`·`FALSE`뿐 아니라 **`UNKNOWN`** 도 가질 수 있습니다. 이를 **3값 논리**라고 합니다. `NULL`은 실제 값을 알 수 없으므로 일반 값과 비교해도 참인지 거짓인지 결정할 수 없습니다.

| 조건식                  | `amount`가 `100000`일 때 | `amount`가 `NULL`일 때 |
| ----------------------- | ------------------------ | ---------------------- |
| `amount >= 50000`       | `TRUE`                   | `UNKNOWN`              |
| `NOT (amount >= 50000)` | `FALSE`                  | `UNKNOWN`              |
| `amount IS NULL`        | `FALSE`                  | `TRUE`                 |
| `amount IS NOT NULL`    | `TRUE`                   | `FALSE`                |

`UNKNOWN`이 논리 연산자와 만나면 다음처럼 판정됩니다. 이미 결과가 정해진 경우에는 `UNKNOWN`이 있어도 결과가 달라지지 않습니다.

| 논리식              | 결과      | 이유                                      |
| ------------------- | --------- | ----------------------------------------- |
| `TRUE AND UNKNOWN`  | `UNKNOWN` | 두 조건이 모두 참인지 알 수 없습니다.     |
| `FALSE AND UNKNOWN` | `FALSE`   | 하나가 이미 거짓이므로 전체도 거짓입니다. |
| `TRUE OR UNKNOWN`   | `TRUE`    | 하나가 이미 참이므로 전체도 참입니다.     |
| `FALSE OR UNKNOWN`  | `UNKNOWN` | 나머지 조건이 참인지 알 수 없습니다.      |

`WHERE`는 판정 결과가 **`TRUE`인 행만** 남기고 `FALSE`와 `UNKNOWN`은 제외합니다. 따라서 `NULL` 여부는 `= NULL`·`<> NULL`이 아니라 **`IS NULL`**·**`IS NOT NULL`** 로 판별합니다.

| 쓰면 안 되는 것 ❌     | 올바른 것 ✅               |
| ---------------------- | -------------------------- |
| `WHERE amount = NULL`  | `WHERE amount IS NULL`     |
| `WHERE amount != NULL` | `WHERE amount IS NOT NULL` |

> ⚠️ **주의하기 — `NOT IN`도 `NULL`의 영향을 받습니다**  
> 대상 열이 `NULL`이면 `NOT IN`의 결과도 `UNKNOWN`이 되어 그 행은 제외됩니다. 비교 목록이나 서브쿼리 결과에 `NULL`이 들어 있어도 예상보다 많은 행이 제외될 수 있습니다. 대상 열의 `NULL`은 바깥 조건에서, 서브쿼리 결과의 `NULL`은 안쪽 쿼리에서 각각 `IS NOT NULL`로 제외합니다.

▶️ **코드 실행하기 · 코드 셀 20 [C20]**  
— **금액이 비어 있는(`NULL`)** 주문만 찾습니다.

In [ ]:
-- [C20]
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount IS NULL

▶️ **코드 실행하기 · 코드 셀 21 [C21]**  
— 반대로 **금액이 입력된** 주문만 남깁니다.

In [ ]:
-- [C21]
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL

> 👀 **결과읽기 — [C21]**  
> `[C20]`의 `IS NULL`은 값이 없는 행을, `[C21]`의 `IS NOT NULL`은 값이 있는 행을 조회합니다. 결측 여부에 따라 분석 대상을 나눌 때 사용하는 기본 조건입니다.

> 🐼 **pandas와 비교**  
> pandas의 `df['amount'].isnull()`·`df['amount'].notnull()`과 같은 역할을 합니다.

## 산술 연산자 — 조회하면서 계산하기

`SELECT`에서 열끼리 또는 열과 숫자로 **계산**할 수 있습니다. (`+`, `-`, `*`, `/`)

예를 들어 금액에 10% 부가세를 더한 값을 바로 계산해 볼 수 있습니다. 다만 `NULL`이 계산에 포함되면 결과도 `NULL`이 됩니다. 예를 들어 `NULL * 1.1`의 결과는 `NULL`입니다. 아래 예제에서 `amount IS NOT NULL`로 금액이 있는 주문만 남기는 이유입니다.

▶️ **코드 실행하기 · 코드 셀 22 [C22]**  
— **금액이 입력된** 주문에 대해, 결제 금액에 **부가세 10%를 더한 값**을 계산해 함께 조회합니다.

In [ ]:
-- [C22]
SELECT
  order_id,
  amount,
  amount * 1.1 AS amount_with_tax
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL

> 👀 **결과읽기 — [C22]**  
> `amount * 1.1`처럼 조회 과정에서 계산식을 적용할 수 있습니다. `AS amount_with_tax`는 계산 결과 열에 별칭을 부여합니다. 별칭은 다음 시간에 자세히 다룹니다.

> 📌 **실무 연결하기 — 산술 연산의 산업별 쓰임**  
> 산술 연산은 할인가(`price * 0.8`), 단위 변환(`bytes / 1024`), 비율(`profit / revenue`) 등을 계산할 때 사용합니다. 나눗셈에서는 분모가 0인지도 함께 확인해야 합니다.

## ⌨️ 백문이 불여일타 (8)

```
[문제] `customers`에서 국가 정보가 `NULL`인 고객을 조회합니다.
```

In [ ]:
-- [C23] ⌨️ 백문이 불여일타 (8) — 국가 정보가 빈 고객 찾기

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 빈 값은 `=`로 찾을 수 없습니다 — `IS NULL`을 씁니다.
- 국가 열은 `customers` 테이블의 `country`입니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  customer_id,
  name,
  country
FROM project_name.dataset_name.customers
WHERE
  country IS NULL
```

</details>

## ⌨️ 백문이 불여일타 (9)

```
[문제] 금액이 있는 주문의 주문번호와 금액의 5%를 계산한 값을 `point`라는 별칭으로 조회합니다.
```

In [ ]:
-- [C24] ⌨️ 백문이 불여일타 (9) — 금액의 5% 적립 포인트 계산

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 5%는 `* 0.05`로, `SELECT` 절에서 바로 계산할 수 있습니다.
- 계산 결과 열에 이름을 붙일 때는 `AS point`처럼 별칭을 씁니다.
- '금액이 있는 주문'만 남기려면 `IS NOT NULL` 조건이 필요합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  amount,
  amount * 0.05 AS point
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
```

</details>

## ⌨️ 백문이 불여일타 (10)

```
[문제] 다음 조건을 모두 만족하는 주문의 조회 쿼리를 완성합니다.
- 상태가 취소(Cancelled)나 반품(Returned)이 아니고
- 금액이 5만원 이상이며
- 금액이 비어 있지 않은 주문
(주문번호·상태·금액을 조회합니다.)
```

In [ ]:
-- [C25] ⌨️ 백문이 불여일타 (10) — 조건 3개를 모두 만족하는 주문 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 취소·반품을 한꺼번에 빼려면 `NOT`과 `IN`을 함께 씁니다.
- 조건 세 개는 `AND`로 이어 붙입니다.
- '비어 있지 않다'는 `IS NOT NULL`이었습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  status NOT IN ('Cancelled', 'Returned')
  AND amount >= 50000
  AND amount IS NOT NULL
```

</details>

## 🚦 짚고 넘어가기

1. `amount = NULL`이 안 되는 이유는 무엇입니까?
2. `NULL`과 숫자 `0`은 같습니까?
3. `WHERE`는 `TRUE`·`FALSE`·`UNKNOWN` 중 어떤 행을 남깁니까?
4. 조회하면서 `amount * 1.1` 같은 계산을 할 수 있습니까? `amount`가 `NULL`이면 결과는 무엇입니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `NULL`은 "값이 없음/모름"이라 일반 값과 비교하면 `TRUE`나 `FALSE`가 아니라 `UNKNOWN`이 되기 때문입니다. `IS NULL`을 사용해야 합니다.
2. 아니요 — `0`은 '값이 0'이라는 정보이고, `NULL`은 값 자체가 없는 상태입니다.
3. `TRUE`인 행만 남깁니다. `FALSE`와 `UNKNOWN`인 행은 모두 제외합니다.
4. 네 — `SELECT` 절에서 `amount * 1.1`처럼 즉석 계산이 가능하고, `AS`로 이름을 붙일 수 있습니다. 다만 `amount`가 `NULL`이면 계산 결과도 `NULL`입니다.

</details>

> ⏭️ **다음 학습 예고**  
> 이제 원하는 열을 선택하고 조건에 맞는 행을 필터링할 수 있습니다. 연습 문제에서 배운 문법을 조합해 봅니다.

# 📒 연습 문제 — 조회와 필터링 (10문항)

지금까지 배운 문법을 반복해서 익힙니다. 아래 10문항의 쿼리를 위에서부터 순서대로 작성하고 실행합니다.

**풀이 방법:** 문제를 먼저 풀고 실행한 뒤 바로 아래의 예시 답안을 펼쳐 비교합니다. 막히면 앞 Part의 예제 셀을 다시 확인합니다.

In [ ]:
-- [C26] 📒 연습 문제 — 조회와 필터링 (1/10)

-- 문제: `status`가 `'Shipped'`인 주문의 `order_id`·`order_date`·`status`를 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [1] 배송 완료 (8행)
SELECT order_id, order_date, status FROM project_name.dataset_name.orders
WHERE status = 'Shipped';
```

</details>

In [ ]:
-- [C27] 📒 연습 문제 — 조회와 필터링 (2/10)

-- 문제: `status`가 `'Cancelled'`가 아닌 주문을 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [2] 취소가 아닌 주문 (27행) — != 를 써도 같습니다
SELECT * FROM project_name.dataset_name.orders
WHERE status <> 'Cancelled';
```

</details>

In [ ]:
-- [C28] 📒 연습 문제 — 조회와 필터링 (3/10)

-- 문제: `order_date`가 2023-10-01 이후인 주문을 조회합니다. 당일을 포함합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [3] 특정 날짜 이후 (24행) — '이후(당일 포함)'이므로 >=
SELECT * FROM project_name.dataset_name.orders
WHERE order_date >= DATE '2023-10-01';
```

</details>

In [ ]:
-- [C29] 📒 연습 문제 — 조회와 필터링 (4/10)

-- 문제: `amount`가 100,000원 미만인 주문의 `order_id`·`order_date`·`amount`를 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [4] 10만원 미만 (14행) — '미만'은 < , '이하'는 <=
SELECT order_id, order_date, amount FROM project_name.dataset_name.orders
WHERE amount < 100000;
```

**확인할 점**

- **미만과 이하** — 한 글자 차이로 경계값 포함 여부가 달라집니다. 문제의 조건을 연산자로 정확히 옮깁니다.

</details>

In [ ]:
-- [C30] 📒 연습 문제 — 조회와 필터링 (5/10)

-- 문제: `amount`가 150,000원 이하이고 `order_date`가 2023-10-01 이후인 주문을 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [5] 두 조건을 모두 만족 (14행)
SELECT * FROM project_name.dataset_name.orders
WHERE amount <= 150000 AND order_date >= DATE '2023-10-01';
```

</details>

In [ ]:
-- [C31] 📒 연습 문제 — 조회와 필터링 (6/10)

-- 문제: `amount`가 100,000원 이하이거나 `order_date`가 2023-11-01 이후인 주문을 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [6] 둘 중 하나라도 만족 (24행)
SELECT * FROM project_name.dataset_name.orders
WHERE amount <= 100000 OR order_date >= DATE '2023-11-01';
```

</details>

In [ ]:
-- [C32] 📒 연습 문제 — 조회와 필터링 (7/10)

-- 문제: `status`가 `'Placed'` 또는 `'Cancelled'`인 주문을 `IN`으로 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [7] 목록에 포함 (6행)
SELECT * FROM project_name.dataset_name.orders
WHERE status IN ('Placed', 'Cancelled');
```

</details>

In [ ]:
-- [C33] 📒 연습 문제 — 조회와 필터링 (8/10)

-- 문제: `status`가 `'Placed'`도 `'Cancelled'`도 아닌 주문을 `NOT IN`으로 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [8] 목록에 없음 (24행)
SELECT * FROM project_name.dataset_name.orders
WHERE status NOT IN ('Placed', 'Cancelled');
```

**확인할 점**

- **`NOT IN`과 `NULL`** — 이 데이터의 `status`에는 `NULL`이 없어서 24행이 조회됩니다. 대상 열이 `NULL`이면 `NOT IN` 결과가 `UNKNOWN`이 되어 그 행은 제외됩니다.

</details>

In [ ]:
-- [C34] 📒 연습 문제 — 조회와 필터링 (9/10)

-- 문제: `amount`가 `NULL`인 주문의 `order_id`·`status`·`amount`를 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [9] 빈 값 찾기 (3행) — = NULL 은 동작하지 않습니다
SELECT order_id, status, amount FROM project_name.dataset_name.orders
WHERE amount IS NULL;
```

**확인할 점**

- **`= NULL`** — `amount = NULL`의 결과는 `UNKNOWN`입니다. `WHERE`에서 참으로 판정되는 행이 없어 0행이 반환되므로 반드시 `IS NULL`을 사용합니다.

</details>

In [ ]:
-- [C35] 📒 연습 문제 — 조회와 필터링 (10/10)

-- 문제: `order_date`가 2023-10-01부터 2023-10-31까지인 주문을 `BETWEEN`으로 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [10] 기간 조회 (6행) — BETWEEN은 양 끝을 포함합니다
SELECT * FROM project_name.dataset_name.orders
WHERE order_date BETWEEN DATE '2023-10-01' AND DATE '2023-10-31';
```

</details>

# 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성

처음 접하는 문법은 낯설 수 있습니다. `SELECT`·`FROM`·`WHERE`와 연산자를 하나씩 확인하며 **"조회 쿼리 모음"**을 완성합니다.

**질문 1.** 2023년 11월에 들어온 주문의 주문번호·주문일·상태·금액을 모두 조회합니다.

In [ ]:
-- [C36] 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성 (1/3)

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  order_date,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  order_date BETWEEN DATE '2023-11-01' AND DATE '2023-11-30'
```

</details>

**질문 2.** 금액이 입력된 주문 중 20만원을 초과하는 고액 주문을 조회합니다.

In [ ]:
-- [C37] 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성 (2/3)

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount > 200000 AND amount IS NOT NULL
```

</details>

**질문 3.** 결제 완료(`Paid`) 또는 배송 완료(`Shipped`) 상태인 주문만 추려 시각화에 사용할 데이터를 준비합니다. 상태별 건수 집계는 다음 시간에 `GROUP BY`와 `COUNT`로 다룹니다.

In [ ]:
-- [C38] 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성 (3/3)

-- 아래에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  status IN ('Paid', 'Shipped')
```

</details>

> 📊 이 결과는 LMS에서는 표로 확인하고, BigQuery에서는 콘솔 차트(또는 Looker Studio)로 시각화할 수 있습니다.

> 👀 **결과읽기 — [C38]**  
> `WHERE status IN ('Paid', 'Shipped')` 조건으로 두 상태의 주문만 남겼습니다. 이 결과는 시각화 도구에 전달할 입력 데이터이며, 아직 상태별 건수를 계산한 결과는 아닙니다.

# 🎯 오늘의 퀴즈

다음 문제로 오늘 배운 내용을 확인합니다. 답을 먼저 작성한 뒤 예시 답안과 비교합니다.

## 개념 퀴즈

1. `SELECT`는 (행 / 열)을 고르고, `WHERE`는 (행 / 열)을 거릅니다. 괄호를 채웁니다.
2. 다음 중 잘못된 것은? ① `WHERE amount >= 100` ② `WHERE status = 'Paid'` ③ `WHERE amount = NULL`
3. `status IN ('Paid', 'Shipped')`와 같은 의미의 조건을 `OR`로 작성합니다.
4. 관계형 데이터베이스가 데이터를 여러 테이블로 나눠 담는 이유는 무엇입니까?
5. `ORDER BY` 없이 `LIMIT 5`를 사용하면 특정한 첫 5행이 항상 반환됩니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `SELECT` = **열**, `WHERE` = **행**
2. **③** — `NULL`은 `=`로 판별하지 않습니다. `IS NULL`을 사용합니다.
3. `status = 'Paid' OR status = 'Shipped'`
4. 같은 정보의 **중복을 줄이고**, 한 곳만 고치면 되도록(유지보수) 하기 위해서입니다.
5. 아니요. SQL 결과의 행 순서는 기본적으로 보장되지 않습니다. 특정 순서의 첫 5행이 필요하면 `ORDER BY`로 순서를 정한 뒤 `LIMIT 5`를 사용해야 합니다.
</details>

## 코드 퀴즈

**문제:** `customers`에서 **국가가 등록되어 있고 미국(USA)이 아니면서**, **등급(`grade`)이 비어 있지 않은** 고객의 이름·국가·등급을 조회합니다.

아래 셀에 직접 작성한 뒤 예시 답안과 비교합니다.

In [ ]:
-- [C39] 코드 퀴즈 — 미국이 아니면서 등급이 있는 고객 조회

-- 여기에 SQL 쿼리를 작성하세요.

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  name,
  country,
  grade
FROM project_name.dataset_name.customers
WHERE
  country IS NOT NULL
  AND country <> 'USA'
  AND grade IS NOT NULL
```

</details>

# 🎓 오늘의 정리

## 오늘 배운 것이 어떻게 이어지는가

```
[Part 1] 데이터베이스·테이블·행·열·스키마·타입  (데이터가 사는 모양)
   ↓  이 표에게 질문하려면…
[Part 2] SELECT        (어떤 '열'을 볼까)
   ↓  전부 말고 조건에 맞는 행만!
[Part 3] WHERE·비교    (어떤 '행'을 거를까)
   ↓  조건이 여러 개라면?
[Part 4] AND·OR·IN·BETWEEN  (조건 조합하기)
   ↓  빈 값과 계산은?
[Part 5] NULL·산술     (IS NULL / 즉석 계산)
   ↓
[다음 시간] 정렬·요약  (ORDER BY·GROUP BY)
```

## 한 장 정리 — 오늘의 쿼리 패턴 치트시트

| 하고 싶은 것         | SQL                                       |
| -------------------- | ----------------------------------------- |
| 모든 열 보기         | `SELECT * FROM 테이블`                    |
| 특정 열만            | `SELECT 열1, 열2 FROM 테이블`             |
| 스키마 보기          | `DESCRIBE 테이블`                         |
| 조건에 맞는 행       | `... WHERE 조건`                          |
| 같다 / 다르다        | `=` / `!=` (또는 `<>`)                    |
| 크기 비교            | `>`, `>=`, `<`, `<=`                      |
| 그리고 / 또는 / 부정 | `AND` / `OR` / `NOT`                      |
| 범위 / 목록          | `BETWEEN a AND b` / `IN (...)`            |
| 빈 값                | `IS NULL` / `IS NOT NULL`                 |
| `NULL`과 비교·계산   | 비교 결과는 `UNKNOWN`, 계산 결과는 `NULL` |
| 계산                 | `+`, `-`, `*`, `/` (예: `amount * 1.1`)   |

> 💡 **핵심짚기 — SQL 질문의 뼈대**  
> `SELECT 열 FROM 테이블 WHERE 조건` — 이 한 문장이 SQL 질문의 뼈대입니다.

## 📝 오늘의 과제

**조회 쿼리 모음 만들기** — 모두몰 데이터에 대한 질문 5개를 만들고 각각을 SQL로 작성합니다.

**제출물(노트북 또는 .sql 파일):**

1. 질문 5개 (예: "Gold 등급인 한국 고객은?")
2. 각 질문에 답하는 SQL 쿼리
3. 각 쿼리 결과에서 발견한 점 1줄

**조건:**

- `WHERE` + 비교/논리 연산자(`AND`/`OR`/`IN`/`BETWEEN`)를 최소 3가지 이상 사용
- `IS NULL` 또는 `IS NOT NULL`을 최소 1번 사용

**평가 기준**

| 항목     | 기준                                  |
| -------- | ------------------------------------- |
| 정확성   | 쿼리가 오류 없이 의도한 결과를 내는가 |
| 다양성   | 여러 연산자를 적절히 활용했는가       |
| 인사이트 | 결과에서 의미 있는 점을 짚었는가      |

**제출:** 개인 GitHub 저장소의 `main` 브랜치에 커밋·푸시한 뒤 링크를 제출합니다.

## ⏭️ 다음 학습 예고 — SQL 기초: 정렬·가공과 집계

오늘은 원하는 행·열을 _고르는_ 법을 배웠습니다. 그런데 결과가 순서 없이 나왔죠?

다음 시간에는 결과를 **정렬**(`ORDER BY`)하고, 중복을 제거(`DISTINCT`)하고, `CASE WHEN`으로 값을 분류하며, 무엇보다 **여러 행을 하나로 요약**(`GROUP BY`·집계함수)하는 법을 배웁니다. _"카테고리별 평균 매출은?"_ 같은 질문에 답할 수 있게 돼요.

오늘 SQL로 데이터에게 처음 말을 걸어봤습니다. 이 작은 첫걸음이 분석가의 가장 강력한 무기가 됩니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>